# location coordinates

the web app has a map, so every location in the house data needs a latitude and longitude. i geocode the names with openstreetmap.

In [1]:
import os, time, json, urllib.parse, urllib.request
import numpy as np
import pandas as pd

houses = pd.read_csv('../data/processed/houses_clean.csv')
locs = houses.groupby(['location', 'district']).size().rename('ads').reset_index()
print('locations:', len(locs))

locations: 202


each name is searched together with its district. nominatim allows one request per second, so this takes a few minutes.

In [2]:
def geocode(query):
    url = 'https://nominatim.openstreetmap.org/search?' + urllib.parse.urlencode(
        {'q': query, 'format': 'json', 'limit': 1, 'countrycodes': 'lk'})
    req = urllib.request.Request(url, headers={'User-Agent': 'lk-ensemble-lab-class-project'})
    with urllib.request.urlopen(req, timeout=30) as r:
        data = json.load(r)
    time.sleep(1.1)
    return (float(data[0]['lat']), float(data[0]['lon'])) if data else (np.nan, np.nan)

rows = []
for _, r in locs.iterrows():
    lat, lon = geocode(f"{r['location']}, {r['district']} District, Sri Lanka")
    rows.append({'location': r['location'], 'district': r['district'], 'latitude': lat, 'longitude': lon})
coords = pd.DataFrame(rows)
coords.to_csv('../data/processed/location_coords.csv', index=False)
print('not found:', coords['latitude'].isna().sum())

not found: 6


6 of the 202 places came back empty. a geocoder can also put a place in the wrong spot, so i still have to check the points.